# Encoding Model Exploration — perceptXbind MEG

Predicts MEG sensor activity from trial-level feature representations.

**Scientific questions**:
- Does stimulus identity (shape one-hot) predict neural activity at rule-phase epochs?
- Does adding role information (shape × role) improve prediction beyond identity alone?
- Where in sensor space and when in time is each feature most predictive?

**Pipeline**:
1. Build feature matrices from behavioral metadata
2. Cross-validate ridge encoding models per sensor per timepoint
3. Map prediction quality (R²) across sensors and time
4. Nested comparison: unique variance of role beyond shape identity

In [ ]:
import sys, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

sys.path.insert(0, str(Path('..').resolve()))
from toolkit import (
    assign_conditions, RULE_PHASES, PHASE_OFFSETS,
    build_feature_space, fit_encoding_model, nested_comparison,
    CVSplitter, EncodingResult,
)

print('toolkit loaded')

## 1. Load data

In [ ]:
EPOCHS_PATH = Path('../ported_results/sub-001_ses-01_task-binding_epo.fif')
BEHAV_PATH  = Path('/mnt/storage/NEU502B/brain-binding/data/2026-04-10/sub-001_events.csv')

epochs = mne.read_epochs(str(EPOCHS_PATH), preload=True)
df     = pd.read_csv(BEHAV_PATH).sort_values('trial').reset_index(drop=True)

X_all  = epochs.get_data(picks='meg').astype(np.float32)  # (1080, n_ch, n_t)
sfreq  = epochs.info['sfreq']
times  = epochs.times
n_ch   = X_all.shape[1]

print(f'Epoch array   : {X_all.shape}')
print(f'n_channels    : {n_ch}')
print(f'Epoch window  : {times[0]:.3f} – {times[-1]:.3f} s  at {sfreq} Hz')

## 2. Feature matrices

In [ ]:
cond_result = assign_conditions(df, scheme='stim_by_role')
ep_phase    = cond_result['epoch_phase']   # (1080,) phase labels

# Build all feature spaces
features = {}
feat_names = {}
for fname in ('stim_identity', 'stim_x_role', 'stim_x_position', 'rule_type', 'combined'):
    Phi, fnames = build_feature_space(df, ep_phase, fname)
    features[fname]   = Phi
    feat_names[fname] = fnames
    print(f'{fname:20s} : shape {Phi.shape}  features: {fnames[:4]}{"..." if len(fnames)>4 else ""}')

In [ ]:
# Restrict to rule-phase epochs (where the feature vectors are non-zero)
rule_mask = np.isin(ep_phase, RULE_PHASES)  # (1080,) bool

X_rule = X_all[rule_mask]   # (360, n_ch, n_t)
Phis   = {k: v[rule_mask] for k, v in features.items()}

print(f'Rule-phase data : {X_rule.shape}')
for k, v in Phis.items():
    print(f'  Phi[{k}] : {v.shape}  non-zero rows: {(v.sum(axis=1)>0).sum()}')

In [ ]:
# Quick sanity: feature matrix heatmap for first 30 rule-phase epochs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, fname in zip(axes, ('stim_identity', 'stim_x_role')):
    ax.imshow(Phis[fname][:30].T, aspect='auto', cmap='Blues', interpolation='none')
    ax.set_xlabel('Epoch (first 30)')
    ax.set_ylabel('Feature dimension')
    ax.set_yticks(range(len(feat_names[fname])))
    ax.set_yticklabels(feat_names[fname], fontsize=8)
    ax.set_title(fname)
plt.suptitle('Feature matrices — rule-phase epochs', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Fit encoding models

In [ ]:
cv = CVSplitter(n_splits=5, stratified=False, shuffle=True, random_state=42)
alpha_grid = np.logspace(-3, 3, 20)

results = {}
for fname in ('stim_identity', 'stim_x_role', 'combined'):
    print(f'Fitting {fname}...', flush=True)
    results[fname] = fit_encoding_model(
        X_rule, Phis[fname], cv,
        feature_names=feat_names[fname],
        alpha_grid=alpha_grid,
        score='r2',
    )
    grand = results[fname].mean_scores.mean()
    print(f'  grand mean R² = {grand:.4f}')

## 4. Time course of mean R² across sensors

In [ ]:
colors_enc = {
    'stim_identity': '#1f77b4',
    'stim_x_role':   '#d62728',
    'combined':      '#2ca02c',
}

fig, ax = plt.subplots(figsize=(11, 4))
for fname, res in results.items():
    # Mean R² across sensors, then mean and SEM across folds
    sensor_mean = res.scores.mean(axis=1)   # (n_folds, n_times)
    m = sensor_mean.mean(axis=0)            # (n_times,)
    s = sensor_mean.std(axis=0) / np.sqrt(sensor_mean.shape[0])
    ax.plot(times, m, label=fname, color=colors_enc[fname], linewidth=1.8)
    ax.fill_between(times, m - s, m + s, alpha=0.15, color=colors_enc[fname])

ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean R² across sensors (mean ± SEM across folds)')
ax.set_title('Encoding model R² — rule-phase epochs')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Nested model comparison: unique role variance

In [ ]:
# Delta R² = stim_x_role - stim_identity
# Positive → role information helps beyond stimulus identity alone
delta = nested_comparison(results['stim_identity'], results['stim_x_role'])
# delta: (n_folds, n_ch, n_t)

delta_mean = delta.mean(axis=0)   # (n_ch, n_t)
delta_sem  = delta.std(axis=0) / np.sqrt(delta.shape[0])

# Time course (mean across sensors)
tc_mean = delta_mean.mean(axis=0)
tc_sem  = delta_sem.mean(axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, tc_mean, color='#9467bd', linewidth=2.0,
        label='ΔR² = stim_x_role − stim_identity')
ax.fill_between(times, tc_mean - tc_sem, tc_mean + tc_sem, alpha=0.2, color='#9467bd')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔR² (mean across sensors)')
ax.set_title('Unique variance of role beyond stimulus identity')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Peak ΔR² = {tc_mean.max():.4f} at t = {times[tc_mean.argmax()]:.3f}s')

## 6. Sensor-space topography at peak time

In [ ]:
# Find peak time for stim_identity model
r2_tc     = results['stim_identity'].mean_scores.mean(axis=0)   # (n_times,)
peak_tidx = r2_tc.argmax()
peak_t    = times[peak_tidx]
print(f'stim_identity R² peaks at t = {peak_t:.3f}s')

# Per-sensor R² at peak
r2_topo_si  = results['stim_identity'].mean_scores[:, peak_tidx]   # (n_ch,)
r2_topo_sxr = results['stim_x_role'].mean_scores[:, peak_tidx]
delta_topo  = delta_mean[:, peak_tidx]

In [ ]:
# MNE topographic plot
meg_picks = mne.pick_types(epochs.info, meg=True, exclude=[])
info_meg  = mne.pick_info(epochs.info, meg_picks)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
topos = [
    (r2_topo_si,  'stim_identity R²',    'RdBu_r'),
    (r2_topo_sxr, 'stim_x_role R²',      'RdBu_r'),
    (delta_topo,  'ΔR² (role unique)',   'PuOr_r'),
]
for ax, (data, title, cmap) in zip(axes, topos):
    vmax = np.percentile(np.abs(data), 97)
    mne.viz.plot_topomap(
        data, info_meg, axes=ax, show=False,
        cmap=cmap, vlim=(-vmax, vmax),
        sensors=False, contours=4,
    )
    ax.set_title(f'{title}\nt = {peak_t:.3f}s', fontsize=9)

plt.suptitle('Sensor-space R² topography at stim_identity peak', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Per-phase comparison: rule1 vs rule2 vs rule3

In [ ]:
phase_results = {}
for phase in RULE_PHASES:
    mask = ep_phase == phase
    X_ph  = X_all[mask]
    Phi_ph = features['stim_identity'][mask]
    print(f'Fitting stim_identity on {phase} ({X_ph.shape[0]} epochs)...')
    phase_results[phase] = fit_encoding_model(
        X_ph, Phi_ph, cv,
        feature_names=feat_names['stim_identity'],
        alpha_grid=alpha_grid, score='r2',
    )
    print(f'  grand mean R² = {phase_results[phase].mean_scores.mean():.4f}')

In [ ]:
phase_colors = {'rule1': '#1f77b4', 'rule2': '#ff7f0e', 'rule3': '#2ca02c'}

fig, ax = plt.subplots(figsize=(11, 4))
for phase in RULE_PHASES:
    res = phase_results[phase]
    m = res.mean_scores.mean(axis=0)
    s = res.sem_scores.mean(axis=0)
    ax.plot(times, m, label=phase, color=phase_colors[phase], linewidth=1.8)
    ax.fill_between(times, m - s, m + s, alpha=0.15, color=phase_colors[phase])

ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean R² across sensors')
ax.set_title('Stimulus identity encoding across rule phases')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Selected alphas

Check that RidgeCV is selecting sensible regularization strengths.

In [ ]:
res_si = results['stim_identity']
# alphas: (n_folds, n_ch, n_times)
mean_alpha_time = res_si.alphas.mean(axis=(0, 1))  # (n_times,)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times, np.log10(mean_alpha_time + 1e-12), color='gray', linewidth=1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('log₁₀(α) — mean across folds and channels')
ax.set_title('RidgeCV selected regularization (stim_identity model)')
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'α range: {mean_alpha_time.min():.3g} – {mean_alpha_time.max():.3g}')
print('(Large α = more regularization; expected when signal-to-noise is low)')

## 9. Summary table

In [ ]:
post_mask = times >= 0.0

rows = []
for fname, res in results.items():
    post_mean = res.mean_scores[:, post_mask].mean()
    all_mean  = res.mean_scores.mean()
    rows.append({'model': fname, 'grand R²': all_mean, 'post-stim R² (0–0.4s)': post_mean,
                 'n_features': Phis[fname].shape[1]})

# Add nested delta
d_post = delta_mean[:, post_mask].mean()
rows.append({'model': 'ΔR² (role unique)', 'grand R²': delta_mean.mean(),
             'post-stim R² (0–0.4s)': d_post, 'n_features': '—'})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format='{:.4f}'.format))

## Interpretation guide

| Result | Interpretation |
|---|---|
| `stim_identity` R² > 0 post-stimulus | MEG encodes which shape is currently shown |
| `stim_x_role` > `stim_identity` | Role information adds predictive power beyond raw shape |
| ΔR² peak timing | When in the epoch does role binding emerge? |
| ΔR² sensor map | Which sensors carry role-binding information? |
| α values | Low α → easy regression problem (high SNR or few features); high α → regularization needed |

**Next steps**:
- Add test-phase epochs and compare rule-phase vs test-phase encoding patterns
- Use `store_weights=True` to visualize which feature dimensions drive each sensor
- Run the same pipeline on TF features (swap `X_rule` for TF-transformed data)
- Feed these results into cross-modal RSA (`cross_modal_rsa`) once fMRI RDMs are available